# MobileNetV3-Small — Modelo final + exportación a TFLite INT8

Entrena el modelo final (con el fix de preprocesamiento ya aplicado), lo exporta a TFLite float32 e INT8, y **verifica el modelo INT8 cuantizado** sobre un conjunto de test 70/15/15.


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install scikit-learn scipy -q

import os, gc, json, shutil, warnings
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import f1_score, classification_report
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Cargar dataset y split 70/15/15 (estratificado)

In [2]:
DATASET_PATH = '/content/drive/MyDrive/tesis_palta/dataset_balanceado'
TFLITE_DIR   = '/content/drive/MyDrive/tesis_palta/modelos_tflite'
os.makedirs(TFLITE_DIR, exist_ok=True)

IMG_SIZE           = (224, 224)
BATCH              = 8
NUM_CLASES         = 3
SEED               = 42
IMAGENES_POR_CLASE = 1000

clases_carpetas = sorted(os.listdir(DATASET_PATH))
print(f"Clases (orden del modelo): {clases_carpetas}")
# Nombres a mostrar (Scab = Roña en español)
NOMBRES = ['Antracnosis', 'Sano', 'Roña']

rutas_todas, etiquetas_todas = [], []
for idx, clase in enumerate(clases_carpetas):
    carpeta = os.path.join(DATASET_PATH, clase)
    for archivo in os.listdir(carpeta):
        if archivo.lower().endswith(('.jpg','.jpeg','.png')):
            rutas_todas.append(os.path.join(carpeta, archivo))
            etiquetas_todas.append(idx)

rutas_todas     = np.array(rutas_todas)
etiquetas_todas = np.array(etiquetas_todas)

np.random.seed(SEED)
indices = []
for c in range(NUM_CLASES):
    idx_c = np.where(etiquetas_todas == c)[0]
    sel = np.random.choice(idx_c, size=min(IMAGENES_POR_CLASE, len(idx_c)), replace=False)
    indices.extend(sel)

rutas     = rutas_todas[np.array(indices)]
etiquetas = etiquetas_todas[np.array(indices)]
rutas, etiquetas = shuffle(rutas, etiquetas, random_state=SEED)

r_train, r_temp, y_train, y_temp = train_test_split(
    rutas, etiquetas, test_size=0.20, stratify=etiquetas, random_state=SEED)
r_val, r_test, y_val, y_test = train_test_split(
    r_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f"Train: {len(r_train)} | Val: {len(r_val)} | Test: {len(r_test)}")
print(f"Distribución test: {np.bincount(y_test)}")

Clases (orden del modelo): ['Antracnosis', 'Roña', 'Sano']
Train: 2400 | Val: 300 | Test: 300
Distribución test: [100 100 100]


## 3. Pipelines tf.data

`mobilenet_v3.preprocess_input` es pasa-todo: el modelo normaliza internamente y recibe [0–255].

In [3]:
def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.mobilenet_v3.preprocess_input(img)
    return img, etiqueta

def aumentar(img, etiqueta):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 30.0)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, etiqueta

def make_ds(r, y, train=False):
    ds = tf.data.Dataset.from_tensor_slices((r, y))
    ds = ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(aumentar, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(500, seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(r_train, y_train, train=True)
ds_val   = make_ds(r_val,   y_val)
ds_test  = make_ds(r_test,  y_test)
print("✅ Datasets con augmentation reforzado")

✅ Datasets con augmentation reforzado


## 4. Construir y entrenar el modelo final (2 fases)

Misma arquitectura y entrenamiento que validaste por k-fold (F1 ≈ 0.888).

In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.keras.backend.clear_session(); gc.collect()

base = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base.trainable = False

entrada = tf.keras.Input(shape=(224, 224, 3))
x = base(entrada, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
salida = layers.Dense(NUM_CLASES, activation='softmax')(x)
modelo_final = models.Model(entrada, salida)

cbs = [EarlyStopping('val_accuracy', patience=8, restore_best_weights=True, verbose=1),
       ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)]

# FASE 1 — base congelada (entrena la cabeza)
modelo_final.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("FASE 1 — base congelada")
modelo_final.fit(ds_train, validation_data=ds_val, epochs=20, callbacks=cbs, verbose=1)

# FASE 2 — fine-tuning amplio: descongela todo MENOS BatchNorm
base.trainable = True
for capa in base.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

modelo_final.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("\nFASE 2 — fine-tuning amplio (BN congelado)")
modelo_final.fit(ds_train, validation_data=ds_val, epochs=30, callbacks=cbs, verbose=1)

print("\n✅ Entrenamiento mejorado completo")

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
FASE 1 — base congelada
Epoch 1/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 360s 870ms/step - accuracy: 0.6954 - loss: 0.7019 - val_accuracy: 0.7867 - val_loss: 0.4947 - learning_rate: 0.0010
Epoch 2/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - accuracy: 0.7779 - loss: 0.5447 - val_accuracy: 0.8300 - val_loss: 0.4023 - learning_rate: 0.0010
Epoch 3/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - accuracy: 0.7875 - loss: 0.5171 - val_accuracy: 0.8400 - val_loss: 0.4051 - learning_rate: 0.0010
Epoch 4/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.7862 - loss: 0.4843 - val_accuracy: 0.8467 - val_loss: 0.3819 - learning_rate: 0.0010
Epoch 5/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 13s 38ms/step - accuracy: 0.8025 - loss: 0.4612 - val_accuracy: 0.8500 - val_loss: 0.3651 - learning_rate: 0.0010
Epoch 6/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - accuracy: 0.8163 - loss: 0.4372 - val_accuracy: 0.8600 - val_loss: 0.3196 - learning_rate: 0.0010
E

## 5. Evaluar el modelo Keras en test (referencia antes de cuantizar)

In [6]:
y_pred, y_true = [], []
for imgs, labs in ds_test:
    y_pred.extend(np.argmax(modelo_final.predict(imgs, verbose=0), axis=1))
    y_true.extend(labs.numpy())
f1_keras = f1_score(y_true, y_pred, average='weighted')
print(f"F1 (Keras, sin cuantizar) en test: {f1_keras:.4f}\n")
print(classification_report(y_true, y_pred, target_names=NOMBRES, digits=4))

F1 (Keras, sin cuantizar) en test: 0.9304

              precision    recall  f1-score   support

 Antracnosis     0.8807    0.9600    0.9187       100
        Sano     0.9894    0.9300    0.9588       100
        Roña     0.9278    0.9000    0.9137       100

    accuracy                         0.9300       300
   macro avg     0.9326    0.9300    0.9304       300
weighted avg     0.9326    0.9300    0.9304       300



## 6. Exportar a TFLite (float32 + INT8)

In [7]:
# Float32 (referencia)
conv = tf.lite.TFLiteConverter.from_keras_model(modelo_final)
ruta_float = f'{TFLITE_DIR}/MobileNetV3Small_float32.tflite'
open(ruta_float,'wb').write(conv.convert())

# INT8 rango dinámico (pesos INT8, activaciones float) — robusto para MobileNetV3
conv = tf.lite.TFLiteConverter.from_keras_model(modelo_final)
conv.optimizations = [tf.lite.Optimize.DEFAULT]   # SIN representative_dataset
ruta_dyn = f'{TFLITE_DIR}/MobileNetV3Small_INT8_dynamic.tflite'
open(ruta_dyn,'wb').write(conv.convert())

# Float16 (respaldo muy robusto)
conv = tf.lite.TFLiteConverter.from_keras_model(modelo_final)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
ruta_f16 = f'{TFLITE_DIR}/MobileNetV3Small_float16.tflite'
open(ruta_f16,'wb').write(conv.convert())

for r in [ruta_float, ruta_dyn, ruta_f16]:
    print(f"{os.path.basename(r)}: {os.path.getsize(r)/(1024*1024):.2f} MB")

Saved artifact at '/tmp/tmpv_hx5g1r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_175')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135814590020880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590020688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590021456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590019920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590021264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590019536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814588777296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814588777104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590021072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135814590019728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1358145887

## 7. Verificar el modelo INT8 cuantizado en test

Confirma que el `.tflite` que vas a desplegar mantiene el rendimiento.

In [8]:
def evaluar_tflite(ruta_modelo):
    interp = tf.lite.Interpreter(
        model_path=ruta_modelo,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
    preds = []
    for ruta in r_test:
        img = tf.io.read_file(ruta)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        img = tf.keras.applications.mobilenet_v3.preprocess_input(img)
        x = tf.expand_dims(img, 0).numpy()
        if inp['dtype'] in (np.int8, np.uint8):
            s, z = inp['quantization']; x = (x/s + z).astype(inp['dtype'])
        else:
            x = x.astype(inp['dtype'])
        interp.set_tensor(inp['index'], x); interp.invoke()
        o = interp.get_tensor(out['index'])[0]
        if out['dtype'] in (np.int8, np.uint8):
            s, z = out['quantization']; o = (o.astype(np.float32)-z)*s
        preds.append(int(np.argmax(o)))
    return f1_score(y_test, preds, average='weighted')

print(f"Keras (sin cuantizar): {f1_keras:.4f}\n")
for nombre, ruta in [('INT8 dinámico', ruta_dyn), ('Float16', ruta_f16)]:
    f1v = evaluar_tflite(ruta)
    tam = os.path.getsize(ruta)/(1024*1024)
    print(f"{nombre:15s}: F1={f1v:.4f} | {tam:.2f} MB")

Keras (sin cuantizar): 0.9304

INT8 dinámico  : F1=0.8624 | 1.14 MB
Float16        : F1=0.9271 | 1.97 MB


## 8. Generar etiquetas.txt (orden correcto del modelo)

In [9]:
ruta_etq = f'{TFLITE_DIR}/etiquetas.txt'
with open(ruta_etq, 'w', encoding='utf-8') as f:
    f.write('\n'.join(NOMBRES))   # Antracnosis, Sano, Roña
print("✅ etiquetas.txt:", NOMBRES)

print("\nArchivos en modelos_tflite/:")
for a in sorted(os.listdir(TFLITE_DIR)):
    print(f"  {a}: {os.path.getsize(os.path.join(TFLITE_DIR,a))/(1024*1024):.2f} MB")

✅ etiquetas.txt: ['Antracnosis', 'Sano', 'Roña']

Archivos en modelos_tflite/:
  MobileNetV3Small_INT8_dynamic.tflite: 1.14 MB
  MobileNetV3Small_float16.tflite: 1.97 MB
  MobileNetV3Small_float32.tflite: 3.87 MB
  etiquetas.txt: 0.00 MB
